# SVM và SVR từ đầu: notebook minh họa (bản gọn, không hình)

Notebook này tái hiện **toàn bộ nội dung bài giảng SVM/SVR (mục 1 đến 8)** bằng mã Python chạy được:
mỗi mục gồm *công thức cốt lõi* (Markdown), *mã cài đặt từ đầu bằng NumPy* và *kết quả so với phép tính tay và scikit-learn*.
Những đoạn mã khó có ô **"Giải thích mã"** ngay sau đó.

**Quy ước** (giống tài liệu): vector đặc trưng $\mathbf{x}_i$ là vector hàng, $\mathbf{w}$ là vector cột, $y_i\in\{-1,+1\}$,
điểm số $s_i=\mathbf{x}_i\mathbf{w}+b$, margin $m_i=y_is_i$, ma trận Gram $\mathbf{K}$, $\mathbf{Q}=\operatorname{diag}(\mathbf{y})\mathbf{K}\operatorname{diag}(\mathbf{y})$.

**Yêu cầu:** `numpy`, `scipy`, `scikit-learn`. Notebook không vẽ hình để giữ gọn; bản `svm_demo.py --plots` và bản HTML có hình minh họa.

| Mục | Nội dung |
|:--:|:--|
| 1 | Margin, khoảng cách có dấu, bài toán lề cực đại |
| 2 | SVM lề cứng: dạng chuẩn tắc, vector hỗ trợ, giải tay |
| 3 | Lề mềm, hinge, gradient dưới đạo hàm, SGD kiểu Pegasos |
| 4 | Đối ngẫu, KKT, ba loại mẫu, tính thưa |
| 5 | Kernel, SMO, XOR bằng kernel đa thức |
| 6 | SVR: ống $\varepsilon$, đối ngẫu, ngoại lai |
| 7 | Đánh giá, Platt scaling, nhiều lớp, grid search |
| 8 | Họ hàm mất mát, lớp mất cân bằng, one-class SVM, xấp xỉ kernel |

In [1]:
import time, warnings
import numpy as np
warnings.filterwarnings('ignore')
np.set_printoptions(precision=4, suppress=True)

def du_lieu_6_sinh_vien():
    """6 sinh viên: x1 = số giờ tự học, x2 = số buổi vắng; y = +1 (đậu), -1 (rớt)."""
    X = np.array([[2, 4], [3, 3], [4, 5], [6, 1], [8, 2], [9, 0]], dtype=float)
    y = np.array([-1, -1, -1, 1, 1, 1], dtype=float)
    return X, y


def du_lieu_7_sinh_vien():
    """Thêm sinh viên 7 (7, 1) bị rớt: nằm lẫn giữa các sinh viên đậu -> KHÔNG tách được."""
    X, y = du_lieu_6_sinh_vien()
    return np.vstack([X, [7, 1]]), np.r_[y, -1.0]


def du_lieu_400_sinh_vien(seed=42):
    """400 sinh viên mô phỏng (giống bài Perceptron / Logistic Regression)."""
    rng = np.random.default_rng(seed)
    n_lop = 200
    dau = np.c_[rng.normal(8, 2.2, n_lop), rng.normal(1.5, 1.2, n_lop)]
    rot = np.c_[rng.normal(4, 2.2, n_lop), rng.normal(3.5, 1.5, n_lop)]
    X = np.clip(np.vstack([dau, rot]), 0, None)
    y = np.r_[np.ones(n_lop), -np.ones(n_lop)]
    return X, y


def chia_train_test(X, y, ti_le_test=0.3, seed=0):
    rng = np.random.default_rng(seed)
    idx = rng.permutation(len(y))
    n_test = int(len(y) * ti_le_test)
    return X[idx[n_test:]], X[idx[:n_test]], y[idx[n_test:]], y[idx[:n_test]]


def chuan_hoa(X_train, *others):
    """Chuẩn hóa theo trung bình / độ lệch chuẩn của TẬP HUẤN LUYỆN (tránh rò rỉ dữ liệu)."""
    mu, sd = X_train.mean(0), X_train.std(0)
    return [(X_train - mu) / sd] + [(Z - mu) / sd for Z in others] + [mu, sd]


def du_lieu_xor():
    X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
    y = np.array([-1, 1, 1, -1], dtype=float)
    return X, y


def du_lieu_sin_co_ngoai_lai(seed=3):
    """y = sin(x) + nhiễu + 3 ngoại lai (dùng cho SVR)."""
    rng = np.random.default_rng(seed)
    x = np.sort(rng.uniform(0, 6, 60))
    y = np.sin(x) + rng.normal(0, 0.12, 60)
    y[[10, 30, 45]] += np.array([1.6, -1.8, 1.5])
    return x[:, None], y

## Mục 1. Từ Perceptron / Logistic Regression đến bài toán lề cực đại

Với mẫu $(\mathbf{x}_i,y_i)$ và siêu phẳng $\mathbf{x}\mathbf{w}+b=0$:

$$m_i=y_i\left(\mathbf{x}_i\mathbf{w}+b\right),\qquad \gamma_i=\frac{m_i}{\lVert\mathbf{w}\rVert},\qquad \gamma(\mathbf{w},b)=\min_i\gamma_i .$$

* **Margin hàm** $m_i$ phụ thuộc thang đo: $(c\mathbf{w},cb)$ cho cùng đường biên nhưng $m_i$ nhân $c$.
* **Khoảng cách có dấu** $\gamma_i$ không phụ thuộc thang đo, nên dùng được để so sánh các đường biên.
* **Bài toán lề cực đại:** $\max_{\mathbf{w},b}\min_i\gamma_i$.

In [2]:
def margin_ham(X, y, w, b):
    """Margin hàm m_i = y_i (x_i w + b). Phụ thuộc thang đo của (w, b)."""
    return y * (X @ w + b)


def khoang_cach_co_dau(X, y, w, b):
    """gamma_i = m_i / ||w||: khoảng cách hình học có dấu, KHÔNG phụ thuộc thang đo."""
    return margin_ham(X, y, w, b) / np.linalg.norm(w)


def le_cua(X, y, w, b):
    """Lề của bộ phân loại = khoảng cách đến mẫu gần nhất = min_i gamma_i."""
    return khoang_cach_co_dau(X, y, w, b).min()

X, y = du_lieu_6_sinh_vien()
w, b = np.array([1.0, -1.0]), -2.5
print("margin hàm       :", margin_ham(X, y, w, b))
print("khoảng cách có dấu:", khoang_cach_co_dau(X, y, w, b).round(3), "-> lề =", round(le_cua(X, y, w, b), 3))
print("nhân (w,b) với 10 :", margin_ham(X, y, 10*w, 10*b), "| khoảng cách vẫn", khoang_cach_co_dau(X, y, 10*w, 10*b).round(3))

# Bốn đường của Perceptron (mục 1.1) so với đường lề cực đại (mục 1.3)
duong = {"A": (-1.0, [4, -8]), "B": (-1.0, [6, -11]), "C": (-1.0, [5, -9]), "D": (0.0, [2, -4]),
         "lề cực đại": (-19/13, [6/13, -4/13])}
for ten, (b_, w_) in duong.items():
    print(f"đường {ten:11s}: lề = {le_cua(X, y, np.array(w_, float), b_):.3f}")

margin hàm       : [4.5 2.5 3.5 2.5 3.5 6.5]
khoảng cách có dấu: [3.182 1.768 2.475 1.768 2.475 4.596] -> lề = 1.768
nhân (w,b) với 10 : [45. 25. 35. 25. 35. 65.] | khoảng cách vẫn [3.182 1.768 2.475 1.768 2.475 4.596]
đường A          : lề = 1.453
đường B          : lề = 1.277
đường C          : lề = 1.263
đường D          : lề = 1.342
đường lề cực đại : lề = 1.803


## Mục 2. SVM tuyến tính lề cứng

Vì $(\mathbf{w},b)\sim(c\mathbf{w},cb)$, ta **chốt thang đo** $\min_i m_i=1$ (dạng chuẩn tắc). Khi đó hai biên lề là $\mathbf{x}\mathbf{w}+b=\pm1$, lề $=1/\lVert\mathbf{w}\rVert$ và bài toán gốc là

$$\min_{\mathbf{w},b}\ \frac12\lVert\mathbf{w}\rVert^2\quad\text{với}\quad y_i\left(\mathbf{x}_i\mathbf{w}+b\right)\ge1 .$$

Dạng ma trận: $\min\frac12\mathbf{u}^T\mathbf{P}\mathbf{u}$ với $\mathbf{G}\mathbf{u}\ge\mathbf{1}$, trong đó $\mathbf{u}=[\mathbf{w};b]$, $\mathbf{G}=[\operatorname{diag}(\mathbf{y})\mathbf{X}\ \ \mathbf{y}]$.

**Cách giải tay (mục 2.5):** đoán hai vector hỗ trợ là *cặp khác lớp gần nhau nhất* $\mathbf{x}_p$ (dương), $\mathbf{x}_q$ (âm).
Khi đó $(\mathbf{x}_p-\mathbf{x}_q)\mathbf{w}=2$ và theo Cauchy–Schwarz, $\lVert\mathbf{w}\rVert$ nhỏ nhất khi $\mathbf{w}$ cùng phương với $\mathbf{x}_p-\mathbf{x}_q$:
$$\mathbf{w}=\frac{2\,(\mathbf{x}_p-\mathbf{x}_q)^T}{\lVert\mathbf{x}_p-\mathbf{x}_q\rVert^2},\qquad b=y_p-\mathbf{x}_p\mathbf{w}.$$
Cuối cùng phải **kiểm tra** mọi ràng buộc còn lại $m_i\ge1$.

In [3]:
def hard_margin_tu_cap_gan_nhat(X, y):
    """Giải tay lề cứng bằng cách ĐOÁN vector hỗ trợ = cặp mẫu khác lớp gần nhau nhất.

    Nếu x_p (dương), x_q (âm) là vector hỗ trợ thì  (x_p - x_q) w = 2.  Theo Cauchy-Schwarz,
    ||w|| nhỏ nhất khi w cùng phương với (x_p - x_q):   w = 2 (x_p - x_q)^T / ||x_p - x_q||^2.
    Sau đó b = y_p - x_p w. Phải KIỂM TRA mọi ràng buộc còn lại m_i >= 1.
    """
    P, N = np.where(y == 1)[0], np.where(y == -1)[0]
    d2 = ((X[P][:, None, :] - X[N][None, :, :]) ** 2).sum(-1)      # bình phương khoảng cách từng cặp
    ip, iq = np.unravel_index(d2.argmin(), d2.shape)
    p, q = P[ip], N[iq]
    diff = X[p] - X[q]
    w = 2 * diff / (diff @ diff)
    b = y[p] - X[p] @ w
    return w, b, (p, q)

w, b, (p, q) = hard_margin_tu_cap_gan_nhat(X, y)
m = margin_ham(X, y, w, b)
print(f"cặp gần nhất: sinh viên {q+1} và {p+1}; w = {w} (= 6/13, -4/13), b = {b:.4f} (= -19/13)")
print("margin:", m.round(3), "| mọi m_i >= 1:", bool((m >= 1 - 1e-9).all()), "| lề =", round(1/np.linalg.norm(w), 4), "(= sqrt(13)/2)")

from sklearn.svm import SVC
ref = SVC(kernel="linear", C=1e8, tol=1e-10).fit(X, y)       # C rất lớn ~ lề cứng
print("sklearn: w =", ref.coef_[0], " b =", ref.intercept_, " vector hỗ trợ:", (ref.support_+1).tolist())

cặp gần nhất: sinh viên 2 và 4; w = [ 0.4615 -0.3077] (= 6/13, -4/13), b = -1.4615 (= -19/13)
margin: [1.769 1.    1.154 1.    1.615 2.692] | mọi m_i >= 1: True | lề = 1.8028 (= sqrt(13)/2)


sklearn: w = [ 0.4615 -0.3077]  b = [-1.4615]  vector hỗ trợ: [2, 4]


**Giải thích mã (`hard_margin_tu_cap_gan_nhat`).**

* `X[P][:, None, :] - X[N][None, :, :]` dùng **phát sóng (broadcasting)**: mảng thứ nhất có hình `(|P|, 1, d)`, mảng thứ hai `(1, |N|, d)`; hiệu của chúng có hình `(|P|, |N|, d)` chứa vector chênh lệch của **mọi cặp** (dương, âm). Bình phương rồi `.sum(-1)` (tổng theo trục cuối) cho ma trận bình phương khoảng cách `d2` cỡ `(|P|, |N|)`, không cần vòng lặp.
* `np.unravel_index(d2.argmin(), d2.shape)` đổi chỉ số phẳng của phần tử nhỏ nhất thành chỉ số (hàng, cột), tức cặp gần nhất.
* `w = 2*diff/(diff @ diff)` chính là công thức $\mathbf{w}=2(\mathbf{x}_p-\mathbf{x}_q)/\lVert\mathbf{x}_p-\mathbf{x}_q\rVert^2$; `b = y[p] - X[p] @ w` suy từ $m_p=1$.
* Hàm **không tự chứng minh tối ưu**: ta kiểm tra $m_i\ge1$ ở bước sau (như phần "đoán, giải, kiểm tra" trong tài liệu).

## Mục 3. SVM lề mềm và hàm mất mát hinge

Thêm biến lỏng $\xi_i\ge0$: $y_i(\mathbf{x}_i\mathbf{w}+b)\ge1-\xi_i$, tối thiểu hóa $\frac12\lVert\mathbf{w}\rVert^2+C\sum_i\xi_i$. Khử $\xi_i=\max(0,1-m_i)$ ta được bài toán **không ràng buộc**

$$J(\mathbf{w},b)=\frac12\lVert\mathbf{w}\rVert^2+C\sum_{i=1}^{n}\max\left(0,\,1-m_i\right).$$

**Dưới đạo hàm** (hinge không khả vi tại $m_i=1$). Đặt $a_i=\mathbb{1}[m_i<1]$ (mẫu vi phạm lề):
$$\nabla_{\mathbf{w}}J=\mathbf{w}-C\,\mathbf{X}^T(\mathbf{y}\odot\mathbf{a}),\qquad \frac{\partial J}{\partial b}=-C\,\mathbf{y}^T\mathbf{a}.$$
Bước cập nhật có dạng *co $\mathbf{w}$ rồi đẩy về phía các mẫu vi phạm*: $\mathbf{w}\leftarrow(1-\eta)\mathbf{w}+\eta C\mathbf{X}^T(\mathbf{y}\odot\mathbf{a})$.
Dùng tốc độ học **giảm dần** để hội tụ; **SGD kiểu Pegasos** chỉ dùng một mẫu mỗi bước với $\eta_t\lambda=1/t$.

In [4]:
class LinearSVM:
    """SVM tuyến tính lề mềm, huấn luyện trên bài toán GỐC không ràng buộc:

        J(w, b) = 1/2 ||w||^2 + C * sum_i max(0, 1 - y_i (x_i w + b))

    method="batch": mỗi bước dùng toàn bộ dữ liệu (gradient dưới đạo hàm).
    method="sgd"  : kiểu Pegasos - mỗi bước chỉ dùng 1 mẫu ngẫu nhiên.
    """

    def __init__(self, C=1.0, n_iter=1000, method="batch", seed=0):
        self.C, self.n_iter, self.method, self.seed = C, n_iter, method, seed

    def objective(self, X, y):
        m = y * (X @ self.w + self.b)
        return 0.5 * self.w @ self.w + self.C * np.maximum(0, 1 - m).sum()

    def fit(self, X, y):
        n, d = X.shape
        # Dạng chuẩn hóa J/(n C) = (lam/2)||w||^2 + trung bình hinge, với lam = 1/(n C).
        # Hàm này lồi chặt với hệ số lam nên bước 1/(lam t) cho đảm bảo hội tụ.
        lam = 1.0 / (n * self.C)
        self.w, self.b, self.history = np.zeros(d), 0.0, []
        rng = np.random.default_rng(self.seed)
        t = 0
        for _ in range(self.n_iter):
            thu_tu = [None] if self.method == "batch" else rng.permutation(n)
            for i in thu_tu:
                t += 1
                eta = 1.0 / (lam * t)                    # tốc độ học GIẢM DẦN
                if i is None:                            # ---- batch
                    # a_i = 1 nếu mẫu i vi phạm lề (m_i < 1), ngược lại 0
                    a = (y * (X @ self.w + self.b) < 1).astype(float)
                    # dưới đạo hàm của f:  lam*w - (1/n) X^T (y ⊙ a)   và   -(1/n) 1^T (y ⊙ a)
                    g_w = lam * self.w - X.T @ (y * a) / n
                    g_b = -np.sum(y * a) / n
                else:                                    # ---- SGD kiểu Pegasos
                    a = float(y[i] * (X[i] @ self.w + self.b) < 1)
                    g_w = lam * self.w - a * y[i] * X[i]
                    g_b = -a * y[i]
                self.w = self.w - eta * g_w
                self.b = self.b - eta * g_b
            self.history.append(self.objective(X, y))
        return self

    def decision_function(self, X):
        return X @ self.w + self.b

    def predict(self, X):
        return np.where(self.decision_function(X) >= 0, 1, -1)


def mot_buoc_batch(X, y, w, b, C, t):
    """MỘT bước Gradient Descent dưới đạo hàm trên J (dạng ở mục 3.4 / 3.6), tốc độ học 1/t."""
    m = y * (X @ w + b)
    a = (m < 1).astype(float)                            # vector chỉ báo "vi phạm lề"
    g_w = w - C * X.T @ (y * a)
    g_b = -C * np.sum(y * a)
    return w - g_w / t, b - g_b / t, m, a

**Giải thích mã (`LinearSVM.fit`).**

* `lam = 1/(n*C)`: ta tối thiểu hóa dạng chuẩn hóa $f=J/(nC)=\frac{\lambda}{2}\lVert\mathbf{w}\rVert^2+\frac1n\sum\text{hinge}$. Hàm này lồi chặt với hệ số $\lambda$ nên bước $\eta_t=1/(\lambda t)$ cho đảm bảo hội tụ; nhân $nC$ vào dưới đạo hàm của $f$ ta được đúng công thức ở trên.
* `a = (y * (X @ w + b) < 1)` là **vector chỉ báo** $\mathbf{a}$ (kiểu bool đổi sang số). `X.T @ (y * a)` là $\mathbf{X}^T(\mathbf{y}\odot\mathbf{a})$: chỉ các mẫu vi phạm lề (đúng nhưng chưa đủ xa, hoặc sai) mới đóng góp.
* `thu_tu = [None]` cho batch (một bước dùng toàn bộ dữ liệu), còn SGD duyệt `rng.permutation(n)` và mỗi bước dùng đúng một mẫu `i`. Với SGD, `history` được ghi sau mỗi **epoch**.
* Giá trị $J$ **không giảm đều** từng bước (dưới đạo hàm không phải hướng giảm). Ở ví dụ tay bên dưới, $J$ đi từ 1,4 lên 4,7 rồi mới giảm dần: đây là bình thường.

In [5]:
X7, y7 = du_lieu_7_sinh_vien()          # thêm sinh viên 7 (7, 1) bị rớt -> không còn tách được
print("Ví dụ tay: 7 sinh viên, C = 0,2, w = 0, b = 0, bước 1/t")
w, b, C_ = np.zeros(2), 0.0, 0.2
for t in range(1, 6):
    J = 0.5 * w @ w + C_ * np.maximum(0, 1 - y7 * (X7 @ w + b)).sum()
    w_moi, b_moi, m, a = mot_buoc_batch(X7, y7, w, b, C_, t)
    print(f"  t={t}: J={J:6.3f}, số mẫu vi phạm={int(a.sum())} -> w={w_moi}, b={b_moi:.3f}")
    w, b = w_moi, b_moi

print("\nẢnh hưởng của C (nghiệm chính xác):  C | w | b | lề | số mẫu có ξ>0")
for C_ in [0.01, 0.1, 1, 10]:
    s = SVC(kernel="linear", C=C_, tol=1e-10).fit(X7, y7)
    w_, b_ = s.coef_[0], s.intercept_[0]
    xi = np.maximum(0, 1 - y7 * (X7 @ w_ + b_))
    print(f"  {C_:5.2f} | {w_.round(3)} | {b_:6.3f} | {1/np.linalg.norm(w_):5.2f} | {(xi > 1e-6).sum()}")

# Huấn luyện tự cài đặt trên dữ liệu ĐÃ CHUẨN HÓA (quan trọng cho tốc độ hội tụ)
Z7, mu, sd = chuan_hoa(X7)
ref = SVC(kernel="linear", C=1, tol=1e-12).fit(Z7, y7)
for method in ["batch", "sgd"]:
    m_ = LinearSVM(C=1, n_iter=1000, method=method).fit(Z7, y7)
    print(f"\nLinearSVM {method:5s}: w = {m_.w}, b = {m_.b:.4f}, J = {m_.history[-1]:.4f}")
print(f"sklearn SVC       : w = {ref.coef_[0]}, b = {ref.intercept_[0]:.4f}")

Ví dụ tay: 7 sinh viên, C = 0,2, w = 0, b = 0, bước 1/t
  t=1: J= 1.400, số mẫu vi phạm=7 -> w=[ 1.4 -2. ], b=-0.200
  t=2: J= 4.700, số mẫu vi phạm=1 -> w=[ 0.  -1.1], b=-0.300
  t=3: J= 2.045, số mẫu vi phạm=3 -> w=[ 1.5333 -0.5333], b=-0.100
  t=4: J= 5.558, số mẫu vi phạm=4 -> w=[ 0.35 -1.05], b=-0.300
  t=5: J= 1.203, số mẫu vi phạm=3 -> w=[ 0.56 -0.76], b=-0.260

Ảnh hưởng của C (nghiệm chính xác):  C | w | b | lề | số mẫu có ξ>0
   0.01 | [ 0.09 -0.06] | -1.000 |  9.25 | 6
   0.10 | [ 0.36 -0.18] | -1.540 |  2.48 | 3
   1.00 | [ 0.346 -0.269] | -1.231 |  2.28 | 2
  10.00 | [ 0.286 -0.571] | -0.143 |  1.57 | 1

LinearSVM batch: w = [ 0.8663 -0.3746], b = 0.0732, J = 2.8011

LinearSVM sgd  : w = [ 0.8673 -0.3736], b = 0.0741, J = 2.8016
sklearn SVC       : w = [ 0.8668 -0.3741], b = 0.0735


## Mục 4. Đối ngẫu và vector hỗ trợ

Từ hàm Lagrange của bài toán lề mềm, điều kiện dừng cho
$$\mathbf{w}=\sum_i\alpha_iy_i\mathbf{x}_i^T=\mathbf{X}^T(\boldsymbol{\alpha}\odot\mathbf{y}),\qquad \mathbf{y}^T\boldsymbol{\alpha}=0,\qquad 0\le\alpha_i\le C,$$
và bài toán **đối ngẫu**
$$\max_{\boldsymbol{\alpha}}\ \mathbf{1}^T\boldsymbol{\alpha}-\tfrac12\boldsymbol{\alpha}^T\mathbf{Q}\boldsymbol{\alpha}\quad\text{với}\quad\mathbf{y}^T\boldsymbol{\alpha}=0,\ \ \mathbf{0}\le\boldsymbol{\alpha}\le C\mathbf{1}.$$

Điều kiện bù trừ (KKT) cho **ba loại mẫu**: $\alpha_i=0\Rightarrow m_i\ge1$; $0<\alpha_i<C\Rightarrow m_i=1$; $\alpha_i=C\Rightarrow m_i\le1$.
Vector hỗ trợ là các mẫu có $\alpha_i>0$; margin được tính qua $\mathbf{m}=\mathbf{Q}\boldsymbol{\alpha}+b\,\mathbf{y}$.

In [6]:
def alpha_tu_sklearn(svm, n):
    """sklearn lưu dual_coef_ = alpha_i * y_i chỉ cho các vector hỗ trợ -> dựng lại alpha đầy đủ."""
    alpha = np.zeros(n)
    alpha[svm.support_] = np.abs(svm.dual_coef_[0])
    return alpha


def perceptron_dang_doi_ngau(X, y, thu_tu, eta=1.0):
    """Perceptron: n_i = số lần mẫu i gây cập nhật. Khi đó w = sum n_i eta y_i x_i, b = sum n_i eta y_i."""
    w, b, dem = np.zeros(X.shape[1]), 0.0, np.zeros(len(y))
    while True:
        loi = 0
        for i in thu_tu:
            if y[i] * (X[i] @ w + b) <= 0:
                w, b, dem[i], loi = w + eta * y[i] * X[i], b + eta * y[i], dem[i] + 1, loi + 1
        if loi == 0:
            return w, b, dem

svm = SVC(kernel="linear", C=1.0, tol=1e-12).fit(X7, y7)
alpha = alpha_tu_sklearn(svm, len(y7))
w, b = svm.coef_[0], svm.intercept_[0]
m = y7 * (X7 @ w + b)
print("alpha =", alpha, " (lý thuyết: 0, 7/26, 0, 1, 7/26, 0, 1)")
for i, a in enumerate(alpha):
    loai = "α=0 (ngoài lề)" if a < 1e-8 else ("α=C (bị chặn)" if a > 1 - 1e-6 else "0<α<C (trên biên lề)")
    print(f"  sinh viên {i+1}: m = {m[i]:6.3f} -> {loai}")

Q = np.outer(y7, y7) * (X7 @ X7.T)                       # Q_ij = y_i y_j x_i x_j^T
print("\nw = Σ α_i y_i x_i =", X7.T @ (alpha * y7), "| sklearn:", w, "| Σ α_i y_i =", round(alpha @ y7, 8))
print("giá trị đối ngẫu =", round(alpha.sum() - 0.5 * alpha @ Q @ alpha, 4),
      "| giá trị gốc =", round(0.5 * w @ w + np.maximum(0, 1 - m).sum(), 4), "(= 127/52)")
print("m = Q α + b y :", Q @ alpha + b * y7)

# Perceptron dạng đối ngẫu: alpha là SỐ LẦN SAI (khác ý nghĩa với alpha của SVM)
_, _, dem = perceptron_dang_doi_ngau(X, y, [0, 1, 2, 3, 4, 5])
print("\nPerceptron đối ngẫu, thứ tự 1..6: số lần cập nhật =", dem.astype(int).tolist())

alpha = [0.     0.2692 0.     1.     0.2692 0.     1.    ]  (lý thuyết: 0, 7/26, 0, 1, 7/26, 0, 1)
  sinh viên 1: m =  1.615 -> α=0 (ngoài lề)
  sinh viên 2: m =  1.000 -> 0<α<C (trên biên lề)
  sinh viên 3: m =  1.192 -> α=0 (ngoài lề)
  sinh viên 4: m =  0.577 -> α=C (bị chặn)
  sinh viên 5: m =  1.000 -> 0<α<C (trên biên lề)
  sinh viên 6: m =  1.885 -> α=0 (ngoài lề)
  sinh viên 7: m = -0.923 -> α=C (bị chặn)

w = Σ α_i y_i x_i = [ 0.3462 -0.2692] | sklearn: [ 0.3462 -0.2692] | Σ α_i y_i = 0.0
giá trị đối ngẫu = 2.4423 | giá trị gốc = 2.4423 (= 127/52)
m = Q α + b y : [ 1.6154  1.      1.1923  0.5769  1.      1.8846 -0.9231]

Perceptron đối ngẫu, thứ tự 1..6: số lần cập nhật = [1, 2, 0, 2, 0, 0]


## Mục 5. SVM phi tuyến: thủ thuật kernel

Dạng đối ngẫu chỉ chứa tích vô hướng $\mathbf{x}_i\mathbf{x}_j^T$. Thay bằng $k(\mathbf{x},\mathbf{z})=\varphi(\mathbf{x})\varphi(\mathbf{z})^T$:
$$f(\mathbf{x})=\sum_{i}\alpha_iy_i\,k(\mathbf{x}_i,\mathbf{x})+b,\qquad Q_{ij}=y_iy_j\,k(\mathbf{x}_i,\mathbf{x}_j).$$
Không cần dựng $\varphi$ (có thể vô hạn chiều). Các kernel phổ biến: tuyến tính $\mathbf{x}\mathbf{z}^T$, đa thức $(\gamma\mathbf{x}\mathbf{z}^T+r)^p$, RBF $e^{-\gamma\lVert\mathbf{x}-\mathbf{z}\rVert^2}$.

**SMO** (giải đối ngẫu): vì $\mathbf{y}^T\boldsymbol{\alpha}=0$ nên phải đổi **cặp** $(\alpha_i,\alpha_j)$: $\alpha_i\leftarrow\alpha_i+y_it$, $\alpha_j\leftarrow\alpha_j-y_jt$.
Hàm đối ngẫu theo $t$ là bậc hai với độ dốc $E_j-E_i$ ($E_k=f(\mathbf{x}_k)-y_k$) và độ cong $\kappa_{ij}=K_{ii}+K_{jj}-2K_{ij}$ nên $t^*=(E_j-E_i)/\kappa_{ij}$, rồi **cắt** để $\alpha$ còn trong $[0,C]$.

In [7]:
def linear_kernel(A, B):
    return A @ B.T


def poly_kernel(A, B, gamma=1.0, coef0=1.0, degree=2):
    return (gamma * (A @ B.T) + coef0) ** degree


def rbf_kernel(A, B, gamma=1.0):
    """Ma trận Gram RBF giữa hai tập điểm KHÔNG DÙNG VÒNG LẶP.

    Dùng khai triển ||a - b||^2 = ||a||^2 + ||b||^2 - 2 a·b:
      - (A**2).sum(1)[:, None]  : cột các ||a_i||^2, cỡ n1 x 1
      - (B**2).sum(1)[None, :]  : hàng các ||b_j||^2, cỡ 1 x n2
      - A @ B.T                 : mọi tích vô hướng a_i·b_j cùng lúc
    NumPy tự "phát sóng" (broadcast) ba số hạng này thành ma trận n1 x n2.
    np.maximum(., 0) chặn sai số làm tròn khiến bình phương khoảng cách hơi âm.
    """
    sq = (A ** 2).sum(1)[:, None] + (B ** 2).sum(1)[None, :] - 2 * A @ B.T
    return np.exp(-gamma * np.maximum(sq, 0))

x, z = np.array([1.0, 2.0]), np.array([2.0, 0.0])
phi = lambda v: np.array([1, np.sqrt(2)*v[0], np.sqrt(2)*v[1], v[0]**2, np.sqrt(2)*v[0]*v[1], v[1]**2])
print("k(x,z) = (x z^T + 1)^2 =", (x @ z + 1) ** 2, "| phi(x)·phi(z) =", round(phi(x) @ phi(z), 6))

k(x,z) = (x z^T + 1)^2 = 9.0 | phi(x)·phi(z) = 9.0


**Giải thích mã (`rbf_kernel`).** Ta cần ma trận $K_{ij}=e^{-\gamma\lVert\mathbf{a}_i-\mathbf{b}_j\rVert^2}$ mà không dùng hai vòng lặp. Khai triển $\lVert\mathbf{a}-\mathbf{b}\rVert^2=\lVert\mathbf{a}\rVert^2+\lVert\mathbf{b}\rVert^2-2\,\mathbf{a}\mathbf{b}^T$:

* `(A**2).sum(1)[:, None]` là cột các $\lVert\mathbf{a}_i\rVert^2$ (hình `n1 x 1`); `(B**2).sum(1)[None, :]` là hàng các $\lVert\mathbf{b}_j\rVert^2$ (hình `1 x n2`);
* `A @ B.T` là **mọi** tích vô hướng $\mathbf{a}_i\mathbf{b}_j^T$ cùng lúc (hình `n1 x n2`);
* NumPy **phát sóng** ba số hạng thành ma trận `n1 x n2`. `np.maximum(sq, 0)` chặn sai số làm tròn làm bình phương khoảng cách hơi âm.

In [8]:
class KernelSVM:
    """SVM lề mềm có kernel, giải bài toán ĐỐI NGẪU bằng SMO đơn giản hóa.

        max  1^T a - 1/2 a^T Q a     với   y^T a = 0,   0 <= a <= C

    Ý tưởng SMO: ràng buộc y^T a = 0 nên không thể đổi MỘT alpha riêng lẻ; ta đổi CẶP (i, j)
        a_i <- a_i + y_i t,    a_j <- a_j - y_j t
    (giữ nguyên tổng y^T a). Khi đó hàm đối ngẫu chỉ còn là hàm bậc hai theo t -> có nghiệm đóng.
    """

    def __init__(self, kernel, C=1.0, tol=1e-3, max_passes=20, seed=0):
        self.kernel, self.C, self.tol = kernel, C, tol
        self.max_passes, self.seed = max_passes, seed

    def fit(self, X, y):
        n = len(y)
        K = self.kernel(X, X)                            # ma trận Gram n x n, tính MỘT lần
        alpha, b = np.zeros(n), 0.0
        rng = np.random.default_rng(self.seed)
        f = lambda i: (alpha * y) @ K[:, i] + b          # hàm quyết định tại mẫu huấn luyện i
        passes = 0
        while passes < self.max_passes:                  # dừng khi một số lượt liên tiếp không đổi gì
            changed = 0
            for i in range(n):
                E_i = f(i) - y[i]                        # sai số E_i = f(x_i) - y_i
                # Mẫu i VI PHẠM điều kiện KKT? (mục 4.3)
                #   y_i f_i < 1 mà alpha_i < C   hoặc   y_i f_i > 1 mà alpha_i > 0
                # (viết qua E_i: y_i f_i - 1 = y_i E_i)
                if (y[i] * E_i < -self.tol and alpha[i] < self.C) or \
                   (y[i] * E_i > self.tol and alpha[i] > 0):
                    j = rng.integers(n - 1)              # chọn j ngẫu nhiên khác i
                    j = j + 1 if j >= i else j
                    E_j = f(j) - y[j]
                    a_i, a_j = alpha[i], alpha[j]
                    # Cận L, H để cả hai alpha vẫn nằm trong [0, C] sau khi đổi (giữ y^T a = 0)
                    if y[i] != y[j]:
                        L, H = max(0, a_j - a_i), min(self.C, self.C + a_j - a_i)
                    else:
                        L, H = max(0, a_i + a_j - self.C), min(self.C, a_i + a_j)
                    kappa = K[i, i] + K[j, j] - 2 * K[i, j]   # độ cong của hàm đối ngẫu theo hướng đổi
                    if L == H or kappa <= 1e-12:
                        continue
                    # Nghiệm đóng của bài toán một biến, rồi CẮT về [L, H]
                    a_j_new = np.clip(a_j + y[j] * (E_i - E_j) / kappa, L, H)
                    if abs(a_j_new - a_j) < 1e-8:
                        continue
                    a_i_new = a_i + y[i] * y[j] * (a_j - a_j_new)   # suy ra alpha_i để giữ y^T a
                    # Cập nhật b sao cho mẫu i (hoặc j) thỏa KKT khớp: ưu tiên mẫu có 0 < alpha < C
                    b1 = b - E_i - y[i] * (a_i_new - a_i) * K[i, i] - y[j] * (a_j_new - a_j) * K[i, j]
                    b2 = b - E_j - y[i] * (a_i_new - a_i) * K[i, j] - y[j] * (a_j_new - a_j) * K[j, j]
                    if 0 < a_i_new < self.C:
                        b = b1
                    elif 0 < a_j_new < self.C:
                        b = b2
                    else:
                        b = (b1 + b2) / 2
                    alpha[i], alpha[j] = a_i_new, a_j_new
                    changed += 1
            passes = passes + 1 if changed == 0 else 0
        self.alpha, self.b = alpha, b
        sv = alpha > 1e-8
        self.X_sv, self.coef = X[sv], (alpha * y)[sv]    # chỉ giữ vector hỗ trợ (tính thưa)
        self.dual_value = alpha.sum() - 0.5 * (alpha * y) @ K @ (alpha * y)
        return self

    def decision_function(self, X):
        # f(x) = sum_i alpha_i y_i k(x_i, x) + b = K_moi (alpha ⊙ y) + b
        return self.kernel(X, self.X_sv) @ self.coef + self.b

    def predict(self, X):
        return np.where(self.decision_function(X) >= 0, 1, -1)

**Giải thích mã (`KernelSVM.fit`, SMO đơn giản hóa).**

1. `K = self.kernel(X, X)` tính **một lần** ma trận Gram $n\times n$; `f(i)` tính $f(\mathbf{x}_i)=\sum_j\alpha_jy_jK_{ji}+b$ chỉ bằng một tích vô hướng.
2. **Kiểm tra vi phạm KKT:** viết $y_iE_i=y_if_i-1$. Mẫu vi phạm nếu $y_if_i<1$ mà $\alpha_i<C$ (`y*E < -tol`), hoặc $y_if_i>1$ mà $\alpha_i>0$ (`y*E > tol`). Chỉ các mẫu vi phạm mới được tối ưu.
3. **Chọn cặp:** `j` ngẫu nhiên khác `i` (bản đơn giản hóa; LIBSVM chọn cặp vi phạm nhiều nhất).
4. **Cận $[L,H]$:** để $\alpha_i,\alpha_j$ còn trong $[0,C]$ sau khi đổi mà giữ $\mathbf{y}^T\boldsymbol{\alpha}=0$. Hai trường hợp `y[i] != y[j]` và ngược lại cho hai cặp cận khác nhau (vì $\alpha_j-\alpha_i$ hoặc $\alpha_i+\alpha_j$ được bảo toàn).
5. **Nghiệm đóng:** `kappa` là độ cong; `a_j_new = a_j + y[j]*(E_i - E_j)/kappa` rồi `np.clip(..., L, H)`. Sau đó `a_i_new = a_i + y[i]*y[j]*(a_j - a_j_new)` để **giữ nguyên** $\sum\alpha_ky_k$.
6. **Cập nhật $b$:** `b1` làm mẫu $i$ thỏa KKT khít, `b2` làm mẫu $j$ thỏa khít; ưu tiên cái nào ứng với $0<\alpha<C$ (vector hỗ trợ tự do), nếu không thì lấy trung bình.
7. Dừng khi `max_passes` lượt liên tiếp không đổi $\alpha$ nào. Cuối cùng chỉ **giữ vector hỗ trợ** (`X_sv`, `coef`$=\alpha_iy_i$), nên dự đoán là `kernel(X_moi, X_sv) @ coef + b`.

In [9]:
def giai_xor_bang_he_kkt():
    """XOR với kernel (x z^T + 1)^2: nếu cả 4 điểm là vector hỗ trợ (m_i = 1) thì
        Q α + b y = 1   và   y^T α = 0   <=>   [[Q, y], [y^T, 0]] [α; b] = [1; 0].
    Đây là hệ 5 phương trình tuyến tính 5 ẩn: giải thẳng bằng np.linalg.solve."""
    X, y = du_lieu_xor()
    K = poly_kernel(X, X, 1.0, 1.0, 2)
    Q = np.outer(y, y) * K
    n = len(y)
    A = np.block([[Q, y[:, None]], [y[None, :], np.zeros((1, 1))]])
    rhs = np.r_[np.ones(n), 0.0]
    sol = np.linalg.solve(A, rhs)
    return K, Q, sol[:n], sol[n]

# (a) XOR giải tay bằng hệ KKT: [[Q, y], [y^T, 0]] [α; b] = [1; 0]  (khi cả 4 điểm là vector hỗ trợ)
K, Q, alpha, b = giai_xor_bang_he_kkt()
Xx, yx = du_lieu_xor()
print("Gram K =\n", K)
print("α =", alpha, "(lý thuyết 10/3, 8/3, 8/3, 2) | b =", b)

# (b) SMO tự cài đặt so với giải tay và sklearn
smo = KernelSVM(lambda A, B: poly_kernel(A, B, 1, 1, 2), C=1e3, max_passes=50).fit(Xx, yx)
print("SMO: α =", smo.alpha.round(3), "b =", round(smo.b, 3), "dự đoán:", smo.predict(Xx))

# (c) Hai vòng tròn có nhiễu: so với sklearn
from sklearn.datasets import make_circles
from sklearn.model_selection import train_test_split
Xc, yc = make_circles(300, noise=0.15, factor=0.45, random_state=1); yc = 2.0 * yc - 1
Xa, Xb, ya, yb = train_test_split(Xc, yc, test_size=0.3, random_state=0)
Za, Zb, _, _ = chuan_hoa(Xa, Xb)
for ten, ker, sk in [("đa thức bậc 2", lambda A, B: poly_kernel(A, B, 1, 1, 2), SVC(kernel="poly", degree=2, gamma=1, coef0=1, C=1)),
                     ("RBF γ=1", lambda A, B: rbf_kernel(A, B, 1.0), SVC(kernel="rbf", gamma=1, C=1))]:
    mine = KernelSVM(ker, C=1.0).fit(Za, ya); sk.fit(Za, ya)
    print(f"{ten:14s} tự cài đặt: test {np.mean(mine.predict(Zb) == yb):.3f}, #SV {len(mine.coef)} | sklearn: test {sk.score(Zb, yb):.3f}, #SV {sk.n_support_.sum()}")

Gram K =
 [[1. 1. 1. 1.]
 [1. 4. 1. 4.]
 [1. 1. 4. 4.]
 [1. 4. 4. 9.]]
α = [3.3333 2.6667 2.6667 2.    ] (lý thuyết 10/3, 8/3, 8/3, 2) | b = -1.0
SMO: α = [3.332 2.665 2.665 1.999] b = -1.0 dự đoán: [-1  1  1 -1]


đa thức bậc 2  tự cài đặt: test 0.944, #SV 30 | sklearn: test 0.944, #SV 30


RBF γ=1        tự cài đặt: test 0.922, #SV 58 | sklearn: test 0.922, #SV 58


## Mục 6. Hồi quy vector hỗ trợ (SVR)

**Mất mát $\varepsilon$-insensitive** của phần dư $r=y-f(\mathbf{x})$: $\ell_\varepsilon(r)=\max(0,\lvert r\rvert-\varepsilon)$ (bằng 0 trong "ống" $\lvert r\rvert\le\varepsilon$, tuyến tính bên ngoài).
$$J(\mathbf{w},b)=\frac12\lVert\mathbf{w}\rVert^2+C\sum_i\max\left(0,\,\lvert y_i-\mathbf{x}_i\mathbf{w}-b\rvert-\varepsilon\right).$$
Dưới đạo hàm: với $s_i=\operatorname{sign}(r_i)\,\mathbb{1}[\lvert r_i\rvert>\varepsilon]$: $\ \nabla_{\mathbf{w}}J=\mathbf{w}-C\mathbf{X}^T\mathbf{s}$, $\ \partial_bJ=-C\mathbf{1}^T\mathbf{s}$.

**Đối ngẫu** với $\boldsymbol{\beta}=\boldsymbol{\alpha}-\boldsymbol{\alpha}^{*}$: $\ \mathbf{w}=\mathbf{X}^T\boldsymbol{\beta}$, $\mathbf{1}^T\boldsymbol{\beta}=0$, $\lvert\beta_i\rvert\le C$ và
$$\max\ -\tfrac12\boldsymbol{\beta}^T\mathbf{K}\boldsymbol{\beta}-\varepsilon\,\mathbf{1}^T(\boldsymbol{\alpha}+\boldsymbol{\alpha}^{*})+\mathbf{y}^T\boldsymbol{\beta}.$$
Mẫu **trong ống** có $\beta_i=0$; **trên biên ống** $0<\lvert\beta_i\rvert<C$; **ngoài ống** $\lvert\beta_i\rvert=C$ (ảnh hưởng của ngoại lai bị chặn).

In [10]:
class LinearSVR:
    """SVR tuyến tính huấn luyện trên bài toán gốc:
        J(w, b) = 1/2 ||w||^2 + C * sum_i max(0, |y_i - x_i w - b| - eps)
    Giống hệt LinearSVM, chỉ khác điều kiện vi phạm (|r_i| > eps) và hướng đẩy sign(r_i) x_i."""

    def __init__(self, C=1.0, eps=0.1, n_iter=5000):
        self.C, self.eps, self.n_iter = C, eps, n_iter

    def objective(self, X, y):
        r = y - X @ self.w - self.b
        return 0.5 * self.w @ self.w + self.C * np.maximum(0, np.abs(r) - self.eps).sum()

    def fit(self, X, y):
        self.w, self.b = np.zeros(X.shape[1]), 0.0
        for t in range(1, self.n_iter + 1):
            r = y - X @ self.w - self.b                  # phần dư
            a = (np.abs(r) > self.eps).astype(float)     # mẫu NGOÀI ống mới bị phạt
            s = np.sign(r) * a                           # s_i ∈ {-1, 0, +1}
            # r_i = y_i - x_i w - b  =>  d r_i / d w = -x_i^T  nên dưới đạo hàm có dấu trừ như dưới
            self.w -= (self.w - self.C * X.T @ s) / t
            self.b -= (-self.C * s.sum()) / t
        return self

    def predict(self, X):
        return X @ self.w + self.b


class KernelSVR:
    """SVR có kernel, giải bài toán đối ngẫu trên 2n biến g = [alpha; alpha*] bằng SLSQP:

        min 1/2 β^T K β + eps 1^T (α + α*) - y^T β,   β = α - α*
        với  1^T β = 0  và  0 <= α, α* <= C

    SLSQP là bộ giải tối ưu có ràng buộc tổng quát của SciPy (đủ cho vài trăm mẫu);
    thư viện thật dùng SMO. Ta cung cấp luôn gradient để bộ giải nhanh và chính xác.
    """

    def __init__(self, kernel, C=1.0, eps=0.1):
        self.kernel, self.C, self.eps = kernel, C, eps

    def fit(self, X, y):
        from scipy.optimize import minimize
        n = len(y)
        K = self.kernel(X, X)

        def obj(g):
            beta = g[:n] - g[n:]
            return 0.5 * beta @ K @ beta + self.eps * g.sum() - y @ beta

        def grad(g):                                     # đạo hàm theo alpha và theo alpha*
            Kb = K @ (g[:n] - g[n:])
            return np.r_[Kb + self.eps - y, -Kb + self.eps + y]

        rang_buoc = {"type": "eq", "fun": lambda g: g[:n].sum() - g[n:].sum(),      # 1^T β = 0
                     "jac": lambda g: np.r_[np.ones(n), -np.ones(n)]}
        res = minimize(obj, np.zeros(2 * n), jac=grad, bounds=[(0, self.C)] * (2 * n),
                       constraints=[rang_buoc], method="SLSQP", options={"maxiter": 500, "ftol": 1e-12})
        beta = res.x[:n] - res.x[n:]
        self.beta = np.where(np.abs(beta) < 1e-7, 0.0, beta)
        sv = self.beta != 0
        self.X_sv, self.coef = X[sv], self.beta[sv]
        # b từ vector hỗ trợ TỰ DO (0 < |β| < C): phần dư của chúng bằng đúng ±eps
        #   y_k - f0_k - b = eps * sign(β_k)  =>  b = y_k - f0_k - eps * sign(β_k)
        tu_do = (np.abs(self.beta) > 1e-7) & (np.abs(self.beta) < self.C - 1e-7)
        f0 = K @ self.beta
        bs = y[tu_do] - f0[tu_do] - self.eps * np.sign(self.beta[tu_do])
        self.b = bs.mean() if tu_do.any() else np.median(y - f0)
        return self

    def predict(self, X):
        return self.kernel(X, self.X_sv) @ self.coef + self.b

**Giải thích mã.**

* `LinearSVR.fit`: `r = y - X@w - b` là phần dư; `a` đánh dấu mẫu **ngoài ống**; `s = sign(r)*a`. Vì $r_i=y_i-\mathbf{x}_i\mathbf{w}-b$ nên $\partial r_i/\partial\mathbf{w}=-\mathbf{x}_i^T$, do đó dưới đạo hàm có dạng `w - C X^T s`. Cập nhật `self.w -= (w - C X^T s)/t` là bước $\eta_t=1/t$ như ở mục 3.
* `KernelSVR.fit` giải bài toán đối ngẫu trên **$2n$ biến** `g = [α; α*]`. `obj` là $\frac12\boldsymbol{\beta}^T\mathbf{K}\boldsymbol{\beta}+\varepsilon\mathbf{1}^T\mathbf{g}-\mathbf{y}^T\boldsymbol{\beta}$ (dạng cực tiểu của hàm đối ngẫu); `grad` là đạo hàm theo $\boldsymbol{\alpha}$ và theo $\boldsymbol{\alpha}^{*}$ (giá trị `Kb` xuất hiện với dấu $+$ và $-$ vì $\boldsymbol{\beta}=\boldsymbol{\alpha}-\boldsymbol{\alpha}^{*}$).
* **SLSQP** là bộ giải tối ưu có ràng buộc của SciPy: ràng buộc đẳng thức `"eq"` là $\mathbf{1}^T\boldsymbol{\beta}=0$ (viết `sum(α) - sum(α*) = 0`), `bounds` là $0\le g_i\le C$. Cung cấp `jac` giúp bộ giải nhanh và chính xác. (SLSQP chỉ hợp cho vài trăm mẫu; thư viện thật dùng SMO.)
* **Tính $b$:** với vector hỗ trợ **tự do** ($0<\lvert\beta_k\rvert<C$) phần dư bằng đúng $\pm\varepsilon$: $y_k-f_k=\varepsilon\,\mathrm{sign}(\beta_k)$, suy ra `b = y - f0 - eps*sign(β)` với `f0 = K @ β`; lấy trung bình để giảm sai số. `beta < 1e-7` được đặt $=0$ để loại bỏ nhiễu số.

In [11]:
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression

x = np.array([1, 2, 3, 4, 5.0]); y = np.array([1.0, 2.2, 2.9, 4.0, 5.0])
print("Hệ số góc khi ngoại lai (x=6, y=y6) di chuyển:   y6 | hồi quy MSE (w, b) | SVR C=1, ε=0,2 (w, b)")
for y6 in [None, 3.0, 2.0, -2.0]:
    xx = x if y6 is None else np.r_[x, 6.0]; yy = y if y6 is None else np.r_[y, y6]
    ols = LinearRegression().fit(xx[:, None], yy)
    svr = LinearSVR(C=1, eps=0.2, n_iter=20000).fit(xx[:, None], yy)
    print(f"   {'-' if y6 is None else y6:>4} | ({ols.coef_[0]:6.3f}, {ols.intercept_:6.3f}) | ({svr.w[0]:6.4f}, {svr.b:6.4f})")
print("(nghiệm tay có ngoại lai: w = 13/15 = 0,8667, b = 1/3 = 0,3333)")

# Kiểm tra KKT / giá trị gốc của nghiệm tay
xx, yy = np.r_[x, 6.0], np.r_[y, 2.0]; w, b, eps = 13/15, 1/3, 0.2
r = yy - (w * xx + b); xi = np.maximum(0, np.abs(r) - eps)
print("\nphần dư:", r.round(3), "| biến lỏng:", xi.round(4), "| J =", round(0.5*w*w + xi.sum(), 4), "(= 1729/450)")

print("\nẢnh hưởng của ε (5 sinh viên sạch, C=1):")
for e in [0.0, 0.2, 0.5, 1.0, 2.0]:
    s = SVR(kernel="linear", C=1, epsilon=e, tol=1e-12).fit(x[:, None], y)
    print(f"  ε={e:3.1f}: w = {s.coef_[0,0]:.3f}, b = {s.intercept_[0]:.3f}, số vector hỗ trợ = {len(s.support_)}")

# SVR RBF trên y = sin(x) có ngoại lai: tự cài đặt (đối ngẫu) so với sklearn
Xs, ys = du_lieu_sin_co_ngoai_lai()
Xs, _, ys, _ = train_test_split(Xs, ys, test_size=0.3, random_state=0)
mu, sd, ym, sy = Xs.mean(), Xs.std(), ys.mean(), ys.std()
Z, zt = (Xs - mu) / sd, (ys - ym) / sy             # chuẩn hóa CẢ x và y (để eps có thang đo hợp lý)
mine = KernelSVR(lambda A, B: rbf_kernel(A, B, 1.0), C=10, eps=0.1).fit(Z, zt)
ref = SVR(kernel="rbf", gamma=1.0, C=10, epsilon=0.1, tol=1e-10).fit(Z, zt)
G = (np.linspace(0, 6, 400)[:, None] - mu) / sd
print(f"\nSVR RBF: số vector hỗ trợ {len(mine.coef)} (tự cài đặt) / {len(ref.support_)} (sklearn); chênh lệch dự đoán lớn nhất {np.abs(mine.predict(G) - ref.predict(G)).max():.1e}")
true = np.sin(np.linspace(0, 6, 400))
print("RMSE so với đường thật sin(x): hồi quy tuyến tính %.3f | SVR RBF %.3f" %
      (np.sqrt(((true - LinearRegression().fit(Xs, ys).predict(np.linspace(0, 6, 400)[:, None]))**2).mean()),
       np.sqrt(((true - (mine.predict(G) * sy + ym))**2).mean())))

Hệ số góc khi ngoại lai (x=6, y=y6) di chuyển:   y6 | hồi quy MSE (w, b) | SVR C=1, ε=0,2 (w, b)


      - | ( 0.980,  0.080) | (0.9002, 0.2998)


    3.0 | ( 0.557,  1.067) | (0.8666, 0.3335)


    2.0 | ( 0.414,  1.400) | (0.8666, 0.3335)


   -2.0 | (-0.157,  2.733) | (0.8666, 0.3335)


(nghiệm tay có ngoại lai: w = 13/15 = 0,8667, b = 1/3 = 0,3333)

phần dư: [-0.2    0.133 -0.033  0.2    0.333 -3.533] | biến lỏng: [0.     0.     0.     0.     0.1333 3.3333] | J = 3.8422 (= 1729/450)

Ảnh hưởng của ε (5 sinh viên sạch, C=1):
  ε=0.0: w = 1.000, b = 0.000, số vector hỗ trợ = 4
  ε=0.2: w = 0.900, b = 0.300, số vector hỗ trợ = 2
  ε=0.5: w = 0.750, b = 0.750, số vector hỗ trợ = 2
  ε=1.0: w = 0.500, b = 1.500, số vector hỗ trợ = 2
  ε=2.0: w = 0.000, b = 3.000, số vector hỗ trợ = 0

SVR RBF: số vector hỗ trợ 26 (tự cài đặt) / 26 (sklearn); chênh lệch dự đoán lớn nhất 1.0e-06
RMSE so với đường thật sin(x): hồi quy tuyến tính 0.422 | SVR RBF 0.137


## Mục 7. Đánh giá, Platt scaling, nhiều lớp, chọn siêu tham số

* **Phân loại:** từ ma trận nhầm lẫn $\begin{bmatrix}TN&FP\\FN&TP\end{bmatrix}$: precision $=\frac{TP}{TP+FP}$, recall $=\frac{TP}{TP+FN}$, $F_1=\frac{2TP}{2TP+FP+FN}$; AUC tính từ **điểm số** $f$.
* **SVM không cho xác suất.** **Platt scaling**: $P(y=+1\mid f)=\sigma(Af+B)$, là Logistic Regression **một chiều** trên điểm số *ngoài mẫu*, tối thiểu hóa cross-entropy bằng Newton.
* **Nhiều lớp:** one-vs-rest ($K$ bộ, $\arg\max f_k$) và one-vs-one ($K(K-1)/2$ bộ, bỏ phiếu).
* **Hồi quy:** $\text{MAE}=\frac1m\sum\lvert r_i\rvert$, $\text{RMSE}=\sqrt{\frac1m\sum r_i^2}$, $R^2=1-\frac{\sum r_i^2}{\sum(y_i-\bar y)^2}$.
* **Chọn $C,\gamma,\varepsilon$** bằng kiểm định chéo trong **pipeline** (chuẩn hóa nằm trong pipeline để không rò rỉ dữ liệu).

In [12]:
def platt_scaling(f, y, n_iter=100):
    """Khớp  P(y=+1 | f) = sigma(A f + B)  bằng phương pháp Newton (Logistic Regression MỘT chiều).

    * Dùng nhãn LÀM MỊN của Platt: t+ = (N+ + 1)/(N+ + 2), t- = 1/(N- + 2) để tránh quá khớp.
    * f phải là điểm số NGOÀI MẪU (ví dụ từ cross_val_predict), nếu không sigmoid sẽ quá dốc.
    Cập nhật Newton:  (A, B) <- (A, B) - H^{-1} g, với
        g = [Σ (p-t) f, Σ (p-t)]      (gradient của cross-entropy)
        H = Σ p(1-p) [[f², f], [f, 1]] (Hessian, nửa xác định dương nên bước Newton ổn định)
    """
    sigma = lambda z: 1 / (1 + np.exp(-z))
    n_pos, n_neg = (y == 1).sum(), (y == -1).sum()
    t = np.where(y == 1, (n_pos + 1) / (n_pos + 2), 1 / (n_neg + 2))
    A, B = 0.0, np.log((n_pos + 1) / (n_neg + 1))
    for _ in range(n_iter):
        p = sigma(A * f + B)
        g = np.array([((p - t) * f).sum(), (p - t).sum()])
        w = p * (1 - p)
        H = np.array([[(w * f * f).sum(), (w * f).sum()], [(w * f).sum(), w.sum()]]) + 1e-9 * np.eye(2)
        buoc = np.linalg.solve(H, g)
        A, B = A - buoc[0], B - buoc[1]
        if np.abs(buoc).max() < 1e-10:
            break
    return A, B

from sklearn.metrics import confusion_matrix, roc_auc_score, log_loss
from sklearn.model_selection import cross_val_predict, StratifiedKFold

X4, y4 = du_lieu_400_sinh_vien()
Xtr, Xte, ytr, yte = chia_train_test(X4, y4)
Ztr, Zte, mu, sd = chuan_hoa(Xtr, Xte)
svm = SVC(kernel="linear", C=1).fit(Ztr, ytr)
f = svm.decision_function(Zte)

(tn, fp), (fn, tp) = confusion_matrix(yte, svm.predict(Zte), labels=[-1, 1])
print(f"[[TN,FP],[FN,TP]] = [[{tn},{fp}],[{fn},{tp}]] | accuracy={(tp+tn)/len(yte):.3f} precision={tp/(tp+fp):.3f} "
      f"recall={tp/(tp+fn):.3f} F1={2*tp/(2*tp+fp+fn):.3f} AUC={roc_auc_score(yte, f):.3f}")

cv = StratifiedKFold(5, shuffle=True, random_state=0)
f_cv = cross_val_predict(SVC(kernel="linear", C=1), Ztr, ytr, cv=cv, method="decision_function")   # điểm số NGOÀI MẪU
A, B = platt_scaling(f_cv, ytr)
sigma = lambda z: 1 / (1 + np.exp(-z)); yb = (yte + 1) // 2
print(f"Platt: A = {A:.4f}, B = {B:.4f} | log-loss: σ(f) ngây thơ {log_loss(yb, sigma(f)):.4f}, Platt {log_loss(yb, sigma(A*f+B)):.4f}")
print(f"ngưỡng theo chi phí (cFP=3, cFN=1): t* = (ln3 - B)/A = {(np.log(3) - B)/A:.3f}")

[[TN,FP],[FN,TP]] = [[60,9],[3,48]] | accuracy=0.900 precision=0.842 recall=0.941 F1=0.889 AUC=0.953
Platt: A = 1.4447, B = 0.0526 | log-loss: σ(f) ngây thơ 0.2758, Platt 0.2781
ngưỡng theo chi phí (cFP=3, cFN=1): t* = (ln3 - B)/A = 0.724


**Giải thích mã (`platt_scaling`).** Ta tối thiểu hóa cross-entropy $-\sum_i[t_i\ln p_i+(1-t_i)\ln(1-p_i)]$ với $p_i=\sigma(Af_i+B)$ theo hai biến $(A,B)$ bằng **phương pháp Newton**:

* gradient `g = [Σ(p-t)f, Σ(p-t)]`; Hessian `H = Σ p(1-p) [[f², f],[f, 1]]` (nửa xác định dương nên bước Newton ổn định); bước `solve(H, g)` rồi `(A,B) -= bước`.
* `t` là **nhãn làm mịn của Platt** ($\frac{N_++1}{N_++2}$ và $\frac1{N_-+2}$) để tránh quá khớp khi chỉ có hai tham số.
* `+ 1e-9*np.eye(2)` thêm một lượng nhỏ vào đường chéo để Hessian luôn khả nghịch.
* Điểm số `f` phải là **ngoài mẫu** (đây là lý do dùng `cross_val_predict`): điểm số của chính tập huấn luyện hầu hết có $f\ge1$ nên sigmoid sẽ quá dốc.

In [13]:
from sklearn.multiclass import OneVsOneClassifier, OneVsRestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV

# ---- Nhiều lớp: yếu / khá / giỏi
rng = np.random.default_rng(7)
means = [(3.5, 4.2), (6.0, 2.6), (8.5, 1.0)]
Xm = np.clip(np.vstack([np.c_[rng.normal(m[0], 1.6, 100), rng.normal(m[1], 1.2, 100)] for m in means]), 0, None)
ym = np.repeat([0, 1, 2], 100)
Xa, Xb, ya, yb3 = train_test_split(Xm, ym, test_size=0.3, random_state=0, stratify=ym)
Za, Zb, _, _ = chuan_hoa(Xa, Xb)
ovr = OneVsRestClassifier(SVC(kernel="linear", C=1)).fit(Za, ya)
ovo = OneVsOneClassifier(SVC(kernel="linear", C=1)).fit(Za, ya)
print(f"3 lớp: One-vs-Rest {ovr.score(Zb, yb3):.3f} | One-vs-One {ovo.score(Zb, yb3):.3f}")
i = 13
diem = [e.decision_function(Zb[i:i+1])[0] for e in ovr.estimators_]
phieu = np.zeros(3)
for (a, b_), e in zip([(0, 1), (0, 2), (1, 2)], ovo.estimators_):
    phieu[b_ if e.decision_function(Zb[i:i+1])[0] > 0 else a] += 1     # >0: nghiêng về lớp thứ hai của cặp
print(f"mẫu {i}: điểm OvR = {np.round(diem, 3)} (đều âm!) -> {int(np.argmax(diem))} | phiếu OvO = {phieu.astype(int).tolist()} -> {int(np.argmax(phieu))} (thực tế {yb3[i]})")

# ---- Chỉ số hồi quy: một dự đoán sai nặng
yt = np.array([2, 3, 5, 7, 9.0])
for ten, yh in [("5 dự đoán tốt", [2.5, 2.8, 5.4, 6.0, 9.5]), ("có dự đoán sai nặng", [2.5, 2.8, 5.4, 6.0, 14.5])]:
    r = yt - np.array(yh)
    print(f"{ten:20s}: MAE={np.abs(r).mean():.3f} RMSE={np.sqrt((r**2).mean()):.3f} R2={1 - (r**2).sum()/((yt-yt.mean())**2).sum():.3f}")

# ---- Grid search RBF trong pipeline (chuẩn hóa nằm BÊN TRONG)
pipe = make_pipeline(StandardScaler(), SVC(kernel="rbf"))
gs = GridSearchCV(pipe, {"svc__C": [0.1, 1, 10, 100], "svc__gamma": [0.001, 0.01, 0.1, 1]}, cv=cv).fit(Xtr, ytr)
print(f"\nGrid search: tốt nhất {gs.best_params_}, CV = {gs.best_score_:.4f}, kiểm tra = {gs.score(Xte, yte):.4f}")

3 lớp: One-vs-Rest 0.789 | One-vs-One 0.811
mẫu 13: điểm OvR = [-1.56  -1.    -1.062] (đều âm!) -> 1 | phiếu OvO = [0, 2, 1] -> 1 (thực tế 1)
5 dự đoán tốt       : MAE=0.520 RMSE=0.583 R2=0.948
có dự đoán sai nặng : MAE=1.520 RMSE=2.518 R2=0.034



Grid search: tốt nhất {'svc__C': 1, 'svc__gamma': 0.01}, CV = 0.9036, kiểm tra = 0.8917


## Mục 8. Tổng kết và mở rộng

**Họ hàm mất mát.** $\ell_{c,\theta}(m)=\frac1c\ln\left(1+e^{c(\theta-m)}\right)$: Logistic ($c=1,\theta=0$), Perceptron ($c\to\infty,\theta=0$), hinge ($c\to\infty,\theta=1$).

**Lớp mất cân bằng:** `class_weight` nhân hệ số phạt $C_i=C\,w_{y_i}$, cận trên đối ngẫu thành $\alpha_i\le C_i$.

**One-class SVM:** $\min\frac12\boldsymbol{\alpha}^T\mathbf{K}\boldsymbol{\alpha}$ với $\mathbf{1}^T\boldsymbol{\alpha}=1$, $0\le\alpha_i\le\frac1{\nu n}$; $\nu$ là cận trên của tỉ lệ vector bị chặn và cận dưới của tỉ lệ vector hỗ trợ.

**Xấp xỉ kernel cho dữ liệu lớn (đặc trưng Fourier ngẫu nhiên):** $k(\mathbf{x},\mathbf{z})\approx\mathbf{z}(\mathbf{x})\mathbf{z}(\mathbf{z})^T$ với $\mathbf{z}(\mathbf{x})=\sqrt{2/D}\cos(\mathbf{x}\boldsymbol{\Omega}+\mathbf{b})$, các cột của $\boldsymbol{\Omega}$ lấy từ $\mathcal{N}(\mathbf{0},2\gamma\mathbf{I})$; sau đó dùng SVM tuyến tính, chi phí $O(nD)$ thay vì $O(n^2)$.

In [14]:
def softplus_dich(m, c, theta):
    """l_{c,θ}(m) = (1/c) ln(1 + exp(c(θ - m))). np.logaddexp(0, z) = ln(1 + e^z) tính ỔN ĐỊNH số học."""
    return np.logaddexp(0, c * (theta - m)) / c


class FourierNgauNhien:
    """Xấp xỉ kernel RBF bằng đặc trưng Fourier ngẫu nhiên (Rahimi & Recht).

    Định lý Bochner: exp(-γ||x-z||²) = E_ω[cos(ω(x-z)^T)] với ω ~ N(0, 2γ I).
    Lấy D mẫu ω rồi dùng đặc trưng  z(x) = sqrt(2/D) cos(xΩ + b)  thì  k(x,z) ≈ z(x)·z(z).
    Sau đó chỉ cần SVM TUYẾN TÍNH trên z(x): chi phí O(nD) thay vì O(n²).
    """

    def __init__(self, gamma=1.0, D=200, seed=0):
        self.gamma, self.D, self.seed = gamma, D, seed

    def fit(self, X):
        rng = np.random.default_rng(self.seed)
        self.Omega = rng.normal(0, np.sqrt(2 * self.gamma), (X.shape[1], self.D))   # các cột ω ~ N(0, 2γI)
        self.b = rng.uniform(0, 2 * np.pi, self.D)
        return self

    def transform(self, X):
        return np.sqrt(2 / self.D) * np.cos(X @ self.Omega + self.b)

**Giải thích mã.**

* `np.logaddexp(0, z)` tính $\ln(1+e^{z})$ **ổn định số học** (không tràn số khi $z$ lớn), nên `softplus_dich` cho kết quả đúng ngay cả khi $c(\theta-m)$ rất lớn.
* `FourierNgauNhien.fit` bốc `Omega` có `d` hàng và `D` cột; mỗi cột là một vector tần số $\boldsymbol{\omega}\sim\mathcal{N}(\mathbf{0},2\gamma\mathbf{I})$ (độ lệch chuẩn `sqrt(2*gamma)`), và `b` là $D$ pha ngẫu nhiên đều trên $[0,2\pi]$. `transform` tính $\sqrt{2/D}\cos(\mathbf{X}\boldsymbol{\Omega}+\mathbf{b})$ cho **mọi mẫu cùng lúc** nhờ phép nhân ma trận + phát sóng của `b`.
* Hệ số $\sqrt{2/D}$ làm $\mathbb{E}[\mathbf{z}(\mathbf{x})\mathbf{z}(\mathbf{z})^T]=k(\mathbf{x},\mathbf{z})$; sai số giảm cỡ $1/\sqrt{D}$ (xem kết quả bên dưới).

In [15]:
from sklearn.svm import LinearSVC, OneClassSVM
from sklearn.metrics import recall_score, precision_score

print("Mất mát (1/c)ln(1+e^{c(1-m)}) co về hinge khi c tăng:")
for m in [0.5, 1.5]:
    print(f"  m={m}: " + ", ".join(f"c={c}: {softplus_dich(m, c, 1.0):.4f}" for c in [1, 5, 20, 100]) + f" | hinge = {max(0, 1 - m)}")

# ---- Lớp mất cân bằng 95/5
rng = np.random.default_rng(1)
Xi = np.vstack([rng.normal([0, 0], 1.2, (950, 2)), rng.normal([1.8, 1.8], 1.0, (50, 2))]); yi = np.r_[-np.ones(950), np.ones(50)]
Xa, Xb, ya, yb = train_test_split(Xi, yi, test_size=0.3, random_state=0, stratify=yi)
Za, Zb, _, _ = chuan_hoa(Xa, Xb)
print("\nLớp mất cân bằng 95/5:")
for ten, cw in [("không trọng số", None), ("balanced", "balanced")]:
    s = SVC(kernel="linear", C=1, class_weight=cw).fit(Za, ya); p = s.predict(Zb)
    print(f"  {ten:15s}: accuracy {np.mean(p == yb):.3f} | recall {recall_score(yb, p):.3f} | precision {precision_score(yb, p):.3f} | AUC {roc_auc_score(yb, s.decision_function(Zb)):.3f}")

# ---- One-class SVM: ν nằm giữa tỉ lệ vector bị chặn và tỉ lệ vector hỗ trợ
Zn, _, _ = chuan_hoa(Xtr[ytr == 1])
print("\nOne-class SVM (RBF γ=0,5) trên sinh viên đậu:   ν | vector bị chặn | vector hỗ trợ")
for nu in [0.05, 0.1, 0.2]:
    o = OneClassSVM(kernel="rbf", gamma=0.5, nu=nu, tol=1e-9).fit(Zn); a = np.abs(o.dual_coef_[0])
    print(f"  {nu:5.2f} | {(a > a.max() * (1 - 1e-6)).sum() / len(Zn):.3f} | {len(o.support_) / len(Zn):.3f}")

# ---- Xấp xỉ kernel bằng Fourier ngẫu nhiên: sai số theo D
x0, z0, gamma = np.zeros(2), np.ones(2), 0.5
k_that = np.exp(-gamma * ((x0 - z0) ** 2).sum())
print(f"\nFourier ngẫu nhiên: k thật = {k_that:.4f}; sai số trung bình qua 20 lần bốc:")
for D in [10, 100, 1000, 10000]:
    sai = [abs((FourierNgauNhien(gamma, D, seed=s).fit(np.zeros((1, 2))).transform(x0[None]) @
                FourierNgauNhien(gamma, D, seed=s).fit(np.zeros((1, 2))).transform(z0[None]).T - k_that).item()) for s in range(20)]
    print(f"  D = {D:6d}: sai số = {np.mean(sai):.4f}")

# ---- Tốc độ: SVC RBF chính xác so với Fourier + LinearSVC (n = 20000)
Xc, yc = make_circles(20000, noise=0.15, factor=0.45, random_state=1)
Xa, Xb, ya, yb = train_test_split(Xc, yc, test_size=0.3, random_state=0)
Za, Zb, _, _ = chuan_hoa(Xa, Xb)
t0 = time.time(); exact = SVC(kernel="rbf", gamma=1, C=1).fit(Za, ya); t1 = time.time() - t0
rf = FourierNgauNhien(1.0, 200).fit(Za)
t0 = time.time(); lin = LinearSVC(C=1, max_iter=100000).fit(rf.transform(Za), ya); t2 = time.time() - t0
print(f"\nn = 20000: SVC RBF {t1:.2f}s (accuracy {exact.score(Zb, yb):.4f}) | Fourier+LinearSVC {t2:.2f}s (accuracy {lin.score(rf.transform(Zb), yb):.4f})")

Mất mát (1/c)ln(1+e^{c(1-m)}) co về hinge khi c tăng:
  m=0.5: c=1: 0.9741, c=5: 0.5158, c=20: 0.5000, c=100: 0.5000 | hinge = 0.5
  m=1.5: c=1: 0.4741, c=5: 0.0158, c=20: 0.0000, c=100: 0.0000 | hinge = 0

Lớp mất cân bằng 95/5:
  không trọng số : accuracy 0.960 | recall 0.333 | precision 0.714 | AUC 0.972
  balanced       : accuracy 0.923 | recall 0.867 | precision 0.382 | AUC 0.972

One-class SVM (RBF γ=0,5) trên sinh viên đậu:   ν | vector bị chặn | vector hỗ trợ
   0.05 | 0.007 | 0.121
   0.10 | 0.060 | 0.154
   0.20 | 0.168 | 0.242

Fourier ngẫu nhiên: k thật = 0.3679; sai số trung bình qua 20 lần bốc:
  D =     10: sai số = 0.3101
  D =    100: sai số = 0.0739
  D =   1000: sai số = 0.0256
  D =  10000: sai số = 0.0091



n = 20000: SVC RBF 0.77s (accuracy 0.9602) | Fourier+LinearSVC 0.31s (accuracy 0.9597)


## Tóm tắt và gợi ý thực hành

* **Ý tưởng cốt lõi:** chọn đường biên có **lề lớn nhất**; hàm mục tiêu lồi cho nghiệm duy nhất; **hinge** chỉ phạt các mẫu có $m<1$ nên nghiệm **thưa** (vector hỗ trợ); **đối ngẫu** cho phép dùng **kernel**; SVR thay lề bằng **ống $\varepsilon$**.
* **Thực hành tốt:** chuẩn hóa đặc trưng trong pipeline; chọn $C,\gamma,\varepsilon$ theo thang log bằng kiểm định chéo; đánh giá bằng nhiều chỉ số; Platt scaling khi cần xác suất; `LinearSVC`/SGD hoặc xấp xỉ kernel khi $n$ lớn.
* **Thử thêm:** đổi $C$, $\gamma$ ở các ô trên và quan sát số vector hỗ trợ; thay dữ liệu bằng `load_breast_cancer`; cài đặt Kernel Ridge Regression $\boldsymbol{\alpha}=(\mathbf{K}+\lambda\mathbf{I})^{-1}\mathbf{y}$ và so với `KernelSVR`.